# Домашняя работа 2. NumPy и базовый Pandas

В работе используются массивы NumPy с логическими масками и базовые возможности Pandas: загрузка таблицы, осмотр, выбор столбцов и строк, фильтрация, `loc` / `iloc`, подсчёт категорий и сортировка. `groupby`, `merge`, pivot-таблицы и циклы не используются.

In [1]:
import numpy as np
import pandas as pd

## Задание 1. NumPy

Даны массивы цен и количеств. Нужно посчитать стоимость позиций, агрегаты и отфильтровать цены с помощью логических масок.

In [2]:
prices = np.array([1200, 800, 2500, 3100, 450, 1800])
counts = np.array([2, 1, 3, 1, 5, 2])

### 1.1–1.3. Поэлементные операции и агрегаты

In [3]:
position_costs = prices * counts
print("Стоимость каждой позиции:", position_costs)

total_cost = position_costs.sum()
print("Общая стоимость всех позиций:", total_cost)

mean_price = prices.mean()
print("Средняя цена:", round(mean_price, 2))

Стоимость каждой позиции: [2400  800 7500 3100 2250 3600]
Общая стоимость всех позиций: 19650
Средняя цена: 1641.67


### 1.4–1.6. Фильтрация логическими масками

Маска — это массив из `True` / `False` той же длины, что и исходный. По маске выбираются только те элементы, где стоит `True`. Поскольку `True` считается как 1, сумма маски даёт количество подходящих элементов.

In [4]:
mask_expensive = prices > 1500
print("Маска prices > 1500:", mask_expensive)
print("Цены больше 1500:", prices[mask_expensive])

Маска prices > 1500: [False False  True  True False  True]
Цены больше 1500: [2500 3100 1800]


In [5]:
mask_range = (prices >= 1000) & (prices <= 3000)
print("Цены от 1000 до 3000 включительно:", prices[mask_range])

expensive_count = mask_expensive.sum()
print("Количество цен больше 1500:", expensive_count)

Цены от 1000 до 3000 включительно: [1200 2500 1800]
Количество цен больше 1500: 3


## Задание 2. Загрузка и первичный осмотр

Загружаем `hw2_orders.csv` в `DataFrame` и смотрим на размер таблицы, столбцы, типы данных и первые / последние строки.

In [6]:
orders = pd.read_csv("hw2_orders.csv")

print("Тип объекта:", type(orders))
print("Строк и столбцов:", orders.shape)
print("Названия столбцов:", list(orders.columns))

Тип объекта: <class 'pandas.DataFrame'>
Строк и столбцов: (30, 8)
Названия столбцов: ['order_id', 'customer', 'city', 'category', 'amount', 'status', 'delivery_days', 'rating']


In [7]:
orders.dtypes

order_id         int64
customer           str
city               str
category           str
amount           int64
status             str
delivery_days    int64
rating           int64
dtype: object

In [8]:
orders.head()

,order_id,customer,city,category,amount,status,delivery_days,rating
0,2001,Дарья Мельникова,Москва,tech,3490,paid,2,5
1,2002,Илья Воронов,Казань,books,990,paid,5,4
2,2003,Полина Жукова,Самара,office,650,paid,4,4
3,2004,Артур Беляев,Москва,home,2290,paid,3,5
4,2005,Елена Фомина,Санкт-Петербург,tech,4890,cancelled,0,0


In [9]:
orders.tail(3)

,order_id,customer,city,category,amount,status,delivery_days,rating
27,2028,Борис Мартынов,Москва,home,2470,paid,2,4
28,2029,Оксана Котова,Казань,office,830,paid,6,4
29,2030,Фёдор Суворов,Самара,tech,6400,paid,1,5


In [10]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_id       30 non-null     int64
 1   customer       30 non-null     str  
 2   city           30 non-null     str  
 3   category       30 non-null     str  
 4   amount         30 non-null     int64
 5   status         30 non-null     str  
 6   delivery_days  30 non-null     int64
 7   rating         30 non-null     int64
dtypes: int64(4), str(4)
memory usage: 2.0 KB


In [11]:
orders.describe()

,order_id,amount,delivery_days,rating
count,30.000000,30.000000,30.00000,30.000000
mean,2015.500000,2509.666667,3.00000,3.700000
std,8.803408,1858.984675,1.83829,1.600646
min,2001.000000,540.000000,0.00000,0.000000
25%,2008.250000,1022.500000,2.00000,4.000000
50%,2015.500000,1960.000000,3.00000,4.000000
75%,2022.750000,3405.000000,4.00000,5.000000
max,2030.000000,7200.000000,6.00000,5.000000


**Числовые столбцы:** `amount` (сумма заказа), `delivery_days` (срок доставки в днях) и `rating` (оценка). `order_id` тоже хранится как число, но это идентификатор заказа: считать по нему среднее или сумму бессмысленно, поэтому его статистика в `describe()` ни о чём не говорит.

**Категориальные столбцы:** `city`, `category` и `status` — в них небольшой набор повторяющихся значений (4 города, 4 категории, 2 статуса). `customer` — текстовый столбец с именами, это скорее идентификатор клиента, чем категория.

`rating` можно рассматривать и как числовой, и как порядковый признак (оценка от 0 до 5). По `describe()` минимальный рейтинг и срок доставки равны 0 — это отменённые заказы, у которых нет ни доставки, ни оценки. Поэтому 0 здесь означает «оценки нет», а не «очень плохая оценка», и это стоит учитывать при расчёте среднего рейтинга.

*Примечание:* в зависимости от версии Pandas текстовые столбцы в `dtypes` отображаются как `object` или `str`.

## Задание 3. Выбор и фильтрация

### 3.1. Выбор столбцов

Один столбец в одинарных скобках возвращает `Series`, список столбцов в двойных скобках — новый `DataFrame`.

In [12]:
amount = orders["amount"]
print("Тип amount:", type(amount))
amount.head()

Тип amount: <class 'pandas.Series'>


0    3490
1     990
2     650
3    2290
4    4890
Name: amount, dtype: int64

In [13]:
orders_short = orders[["customer", "city", "category", "amount"]]
print("Тип orders_short:", type(orders_short))
orders_short.head()

Тип orders_short: <class 'pandas.DataFrame'>


,customer,city,category,amount
0,Дарья Мельникова,Москва,tech,3490
1,Илья Воронов,Казань,books,990
2,Полина Жукова,Самара,office,650
3,Артур Беляев,Москва,home,2290
4,Елена Фомина,Санкт-Петербург,tech,4890


### 3.2. Фильтрация по одному условию

In [14]:
paid_orders = orders[orders["status"] == "paid"]
print("Оплаченных заказов:", len(paid_orders))
paid_orders.head()

Оплаченных заказов: 26


,order_id,customer,city,category,amount,status,delivery_days,rating
0,2001,Дарья Мельникова,Москва,tech,3490,paid,2,5
1,2002,Илья Воронов,Казань,books,990,paid,5,4
2,2003,Полина Жукова,Самара,office,650,paid,4,4
3,2004,Артур Беляев,Москва,home,2290,paid,3,5
5,2006,Кирилл Громов,Казань,home,1750,paid,6,3


In [15]:
moscow_orders = orders[orders["city"] == "Москва"]
print("Заказов из Москвы:", len(moscow_orders))
moscow_orders

Заказов из Москвы: 10


,order_id,customer,city,category,amount,status,delivery_days,rating
0,2001,Дарья Мельникова,Москва,tech,3490,paid,2,5
3,2004,Артур Беляев,Москва,home,2290,paid,3,5
6,2007,Валерия Тихонова,Москва,books,1350,paid,2,5
9,2010,Глеб Макаров,Москва,tech,2600,paid,1,5
12,2013,Ксения Миронова,Москва,tech,7200,paid,2,5
15,2016,Матвей Соловьёв,Москва,office,760,paid,3,3
18,2019,Татьяна Фролова,Москва,home,1940,paid,2,4
21,2022,Владимир Гусев,Москва,books,1680,paid,5,4
24,2025,Нина Данилова,Москва,office,610,cancelled,0,0
27,2028,Борис Мартынов,Москва,home,2470,paid,2,4


In [16]:
expensive_orders = orders[orders["amount"] > 3000]
print("Заказов дороже 3000:", len(expensive_orders))
expensive_orders

Заказов дороже 3000: 9


,order_id,customer,city,category,amount,status,delivery_days,rating
0,2001,Дарья Мельникова,Москва,tech,3490,paid,2,5
4,2005,Елена Фомина,Санкт-Петербург,tech,4890,cancelled,0,0
7,2008,Олег Комаров,Самара,tech,5790,paid,3,4
12,2013,Ксения Миронова,Москва,tech,7200,paid,2,5
14,2015,Людмила Баранова,Казань,tech,3150,paid,4,4
19,2020,Павел Казаков,Казань,tech,4250,paid,3,5
22,2023,Маргарита Орехова,Санкт-Петербург,tech,5350,paid,2,5
25,2026,Ярослав Крылов,Самара,tech,3800,paid,3,4
29,2030,Фёдор Суворов,Самара,tech,6400,paid,1,5


In [17]:
top_rated_orders = orders[orders["rating"] == 5]
print("Заказов с рейтингом 5:", len(top_rated_orders))
top_rated_orders

Заказов с рейтингом 5: 10


,order_id,customer,city,category,amount,status,delivery_days,rating
0,2001,Дарья Мельникова,Москва,tech,3490,paid,2,5
3,2004,Артур Беляев,Москва,home,2290,paid,3,5
6,2007,Валерия Тихонова,Москва,books,1350,paid,2,5
9,2010,Глеб Макаров,Москва,tech,2600,paid,1,5
12,2013,Ксения Миронова,Москва,tech,7200,paid,2,5
16,2017,Вероника Власова,Самара,books,1120,paid,6,5
19,2020,Павел Казаков,Казань,tech,4250,paid,3,5
22,2023,Маргарита Орехова,Санкт-Петербург,tech,5350,paid,2,5
26,2027,Алёна Зайцева,Санкт-Петербург,books,1250,paid,5,5
29,2030,Фёдор Суворов,Самара,tech,6400,paid,1,5


### 3.3. Фильтрация по нескольким условиям

Условия объединяются через `&` (И) и `|` (ИЛИ), каждое условие берётся в круглые скобки.

In [18]:
paid_moscow_2000 = orders[
    (orders["status"] == "paid")
    & (orders["city"] == "Москва")
    & (orders["amount"] > 2000)
]
print("Оплаченных московских заказов дороже 2000:", len(paid_moscow_2000))
paid_moscow_2000

Оплаченных московских заказов дороже 2000: 5


,order_id,customer,city,category,amount,status,delivery_days,rating
0,2001,Дарья Мельникова,Москва,tech,3490,paid,2,5
3,2004,Артур Беляев,Москва,home,2290,paid,3,5
9,2010,Глеб Макаров,Москва,tech,2600,paid,1,5
12,2013,Ксения Миронова,Москва,tech,7200,paid,2,5
27,2028,Борис Мартынов,Москва,home,2470,paid,2,4


In [19]:
kazan_or_samara = orders[(orders["city"] == "Казань") | (orders["city"] == "Самара")]
print("Заказов из Казани или Самары:", len(kazan_or_samara))
kazan_or_samara

Заказов из Казани или Самары: 14


,order_id,customer,city,category,amount,status,delivery_days,rating
1,2002,Илья Воронов,Казань,books,990,paid,5,4
2,2003,Полина Жукова,Самара,office,650,paid,4,4
5,2006,Кирилл Громов,Казань,home,1750,paid,6,3
7,2008,Олег Комаров,Самара,tech,5790,paid,3,4
10,2011,Марина Алексеева,Казань,office,540,cancelled,0,0
11,2012,Степан Захаров,Самара,home,1980,paid,5,4
14,2015,Людмила Баранова,Казань,tech,3150,paid,4,4
16,2017,Вероника Власова,Самара,books,1120,paid,6,5
19,2020,Павел Казаков,Казань,tech,4250,paid,3,5
20,2021,Арина Маслова,Самара,office,920,paid,4,4


In [20]:
tech_rating_5 = orders[(orders["category"] == "tech") & (orders["rating"] == 5)]
print("Заказов категории tech с рейтингом 5:", len(tech_rating_5))
tech_rating_5

Заказов категории tech с рейтингом 5: 6


,order_id,customer,city,category,amount,status,delivery_days,rating
0,2001,Дарья Мельникова,Москва,tech,3490,paid,2,5
9,2010,Глеб Макаров,Москва,tech,2600,paid,1,5
12,2013,Ксения Миронова,Москва,tech,7200,paid,2,5
19,2020,Павел Казаков,Казань,tech,4250,paid,3,5
22,2023,Маргарита Орехова,Санкт-Петербург,tech,5350,paid,2,5
29,2030,Фёдор Суворов,Самара,tech,6400,paid,1,5


### 3.4. `isin()` и `between()`

Город — Москва, Казань или Санкт-Петербург, сумма от 1500 до 4000 включительно (`between()` по умолчанию включает обе границы).

In [21]:
city_amount_filtered = orders[
    orders["city"].isin(["Москва", "Казань", "Санкт-Петербург"])
    & orders["amount"].between(1500, 4000)
]
print("Подходящих заказов:", len(city_amount_filtered))
city_amount_filtered

Подходящих заказов: 10


,order_id,customer,city,category,amount,status,delivery_days,rating
0,2001,Дарья Мельникова,Москва,tech,3490,paid,2,5
3,2004,Артур Беляев,Москва,home,2290,paid,3,5
5,2006,Кирилл Громов,Казань,home,1750,paid,6,3
9,2010,Глеб Макаров,Москва,tech,2600,paid,1,5
14,2015,Людмила Баранова,Казань,tech,3150,paid,4,4
17,2018,Денис Куликов,Санкт-Петербург,home,2840,cancelled,0,0
18,2019,Татьяна Фролова,Москва,home,1940,paid,2,4
21,2022,Владимир Гусев,Москва,books,1680,paid,5,4
23,2024,Антон Быков,Казань,home,2050,paid,4,3
27,2028,Борис Мартынов,Москва,home,2470,paid,2,4


## Задание 4. `loc`, `iloc`, категории и сортировка

### 4.1. `iloc` — выбор по позиции

Первые 5 строк и первые 4 столбца. В `iloc` правая граница среза не включается.

In [22]:
orders.iloc[:5, :4]

,order_id,customer,city,category
0,2001,Дарья Мельникова,Москва,tech
1,2002,Илья Воронов,Казань,books
2,2003,Полина Жукова,Самара,office
3,2004,Артур Беляев,Москва,home
4,2005,Елена Фомина,Санкт-Петербург,tech


### 4.2. `loc` — выбор по меткам

Строки с индексами от 0 до 4 и столбцы `customer`, `city`, `amount`. В `loc` правая граница среза **включается**, поэтому пишем `0:4`.

In [23]:
orders.loc[0:4, ["customer", "city", "amount"]]

,customer,city,amount
0,Дарья Мельникова,Москва,3490
1,Илья Воронов,Казань,990
2,Полина Жукова,Самара,650
3,Артур Беляев,Москва,2290
4,Елена Фомина,Санкт-Петербург,4890


### 4.3. Категории

In [24]:
unique_cities = orders["city"].unique()
print("Уникальные города:", unique_cities)
print("Количество уникальных городов:", orders["city"].nunique())

Уникальные города: <StringArray>
['Москва', 'Казань', 'Самара', 'Санкт-Петербург']
Length: 4, dtype: str
Количество уникальных городов: 4


In [25]:
orders["category"].value_counts()

category
tech      10
office     7
home       7
books      6
Name: count, dtype: int64

In [26]:
orders["status"].value_counts()

status
paid         26
cancelled     4
Name: count, dtype: int64

### 4.4. Сортировка

Оставляем оплаченные заказы и нужные столбцы, сортируем по сумме по убыванию и сбрасываем индекс.

In [27]:
paid_sorted = (
    orders.loc[
        orders["status"] == "paid",
        ["order_id", "customer", "city", "category", "amount"],
    ]
    .sort_values("amount", ascending=False)
    .reset_index(drop=True)
)

paid_sorted.head(10)

,order_id,customer,city,category,amount
0,2013,Ксения Миронова,Москва,tech,7200
1,2030,Фёдор Суворов,Самара,tech,6400
2,2008,Олег Комаров,Самара,tech,5790
3,2023,Маргарита Орехова,Санкт-Петербург,tech,5350
4,2020,Павел Казаков,Казань,tech,4250
5,2026,Ярослав Крылов,Самара,tech,3800
6,2001,Дарья Мельникова,Москва,tech,3490
7,2015,Людмила Баранова,Казань,tech,3150
8,2010,Глеб Макаров,Москва,tech,2600
9,2028,Борис Мартынов,Москва,home,2470


## Задание 5. Краткий вывод

**1. Чем `Series` отличается от `DataFrame`?**
`Series` — это один столбец: одномерный набор значений с индексом (например, `orders["amount"]`). `DataFrame` — это таблица из нескольких столбцов, у которых общий индекс; каждый столбец `DataFrame` сам является `Series`. Поэтому `orders["amount"]` вернул `Series`, а `orders[["customer", "city", "category", "amount"]]` — `DataFrame`.

**2. В чём разница между `loc` и `iloc`?**
`iloc` выбирает строки и столбцы по их порядковым номерам, а `loc` — по меткам: значениям индекса и названиям столбцов. В `iloc[:5]` пятая позиция не включается, а в `loc[0:4]` метка 4 включается, поэтому оба выражения в задании 4 вернули по 5 строк.

**3. Как связаны логические маски NumPy и фильтрация Pandas?**
Это один и тот же механизм. Сравнение `prices > 1500` в NumPy и `orders["amount"] > 3000` в Pandas даёт набор `True` / `False`, и по нему выбираются элементы или строки, где стоит `True`. Условия так же объединяются через `&` и `|`, а сумма маски в обоих случаях считает количество подходящих элементов.

**4. Какой запрос оказался самым сложным и почему?**
Фильтрация по нескольким условиям: легко забыть скобки вокруг каждого условия или написать `and` / `or` вместо `&` / `|`, и тогда Pandas выдаёт ошибку. Также пришлось внимательно следить за границами: `between()` включает обе границы, а срезы `loc` и `iloc` ведут себя по-разному.